In [ ]:
import os
import psycopg2
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

file_path = "Taxi_Trip_Data.csv"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

table_name = "taxi_trips"


def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


chunksize = 500000
first_chunk = True

for chunk in pd.read_csv(file_path, chunksize=chunksize, low_memory=False,
                         parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime']):
    
    
    chunk = chunk.loc[:, ~chunk.columns.str.contains('^Unnamed')]
    
    
    chunk.columns = [col.strip().replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct")
                     for col in chunk.columns]

    
    if first_chunk:
        columns = chunk.dtypes
        sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])
        create_stmt = f'CREATE TABLE "{table_name}" (\n  {sql_columns}\n);'

        cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
        cur.execute(create_stmt)
        conn.commit()
        print("Table created with schema:")
        print(create_stmt)
        first_chunk = False

    
    columns_list = list(chunk.columns)
    placeholders = ', '.join(['%s'] * len(columns_list))
    quoted_cols = ', '.join([f'"{col}"' for col in columns_list])
    insert_stmt = f'INSERT INTO "{table_name}" ({quoted_cols}) VALUES ({placeholders})'

    for _, row in chunk.iterrows():
        row_values = [None if pd.isna(val) else val for val in row[columns_list]]
        cur.execute(insert_stmt, tuple(row_values))

    conn.commit()
    print(f"Inserted chunk of {len(chunk)} rows successfully")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}" LIMIT 5', conn)
print(df_from_db)

cur.close()
conn.close()


Table created with schema:
CREATE TABLE "taxi_trips" (
  "VendorID" FLOAT,
  "tpep_pickup_datetime" TIMESTAMP,
  "tpep_dropoff_datetime" TIMESTAMP,
  "passenger_count" FLOAT,
  "trip_distance" FLOAT,
  "RatecodeID" FLOAT,
  "store_and_fwd_flag" TEXT,
  "PULocationID" INT,
  "DOLocationID" INT,
  "payment_type" FLOAT,
  "fare_amount" FLOAT,
  "extra" FLOAT,
  "mta_tax" FLOAT,
  "tip_amount" FLOAT,
  "tolls_amount" FLOAT,
  "improvement_surcharge" FLOAT,
  "total_amount" FLOAT,
  "congestion_surcharge" FLOAT
);
Inserted chunk of 500000 rows successfully
Inserted chunk of 500000 rows successfully
